# Transformação Bronze → Silver: Suppliers

## 🎯 Objetivo
Transformar a tabela `bronze.bronze_suppliers` aplicando limpeza, padronização de tipos e validações de qualidade para gerar a tabela `silver.suppliers`.

## 📊 Diferenças em Relação a Customers
- **SEM SCD Type 2**: Suppliers não tem versionamento (valid_from/valid_to)
- **Coluna específica**: items_provided (lista de itens fornecidos)
- **Mais simples**: Sem métricas de compra ou loyalty

## 📦 Estrutura da Tabela
- **SUPPLIER_ID**: INT (chave primária)
- **TAX_ID**: INT (identificador fiscal)
- **supplier_name**: STRING (nome do fornecedor)
- **Endereço**: state, city, postcode, street, number, unit, region, district
- **Coordenadas**: lon, lat
- **items_provided**: STRING (itens fornecidos)

## 🔄 Transformações Planejadas
1. Limpeza e padronização de strings
2. Conversão de tipos de dados (postcode: DOUBLE → STRING)
3. Remoção de duplicados completos
4. Validação de coordenadas
5. Adição de colunas de auditoria
6. Validações de qualidade

---

In [0]:
# Imports necessários
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
import uuid

# Configuração de variáveis
CATALOG = "retail_dev"
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
BRONZE_TABLE = "bronze_suppliers"
SILVER_TABLE = "suppliers"

# Identificador único desta execução para auditoria
pipeline_run_id = str(uuid.uuid4())
processing_timestamp = datetime.now()

print(f"🔧 Configuração concluída")
print(f"📌 Pipeline Run ID: {pipeline_run_id}")
print(f"⏰ Timestamp de Processamento: {processing_timestamp}")
print(f"📥 Origem: {CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
print(f"📤 Destino: {CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}")

# 📖 Etapa 1: Leitura da Tabela Bronze

Carregar os dados da camada bronze e realizar uma análise inicial.

In [0]:
# Ler a tabela bronze
bronze_df = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")

# Contagem inicial
initial_count = bronze_df.count()

print(f"✅ Tabela bronze carregada com sucesso!")
print(f"📈 Total de registros: {initial_count:,}")
print(f"\n📊 Schema da tabela bronze:")
bronze_df.printSchema()

# Exibir amostra dos dados
print(f"\n🔍 Amostra dos dados (5 primeiras linhas):")
display(bronze_df.limit(5))

# 🔍 Etapa 2: Análise de Qualidade (PRÉ-TRANSFORMAÇÃO)

Auditoria dos dados antes de aplicar transformações:
- Identificar duplicados
- Contar valores nulos
- Analisar problemas específicos em cada coluna

In [0]:
# 🔵 AUDITORIA: Contagem de duplicados por SUPPLIER_ID
duplicates_df = bronze_df.groupBy("SUPPLIER_ID").count().filter(F.col("count") > 1)
duplicates_count = duplicates_df.count()
total_duplicate_records = duplicates_df.agg(F.sum("count")).collect()[0][0] if duplicates_count > 0 else 0

print(f"🔴 DUPLICADOS ENCONTRADOS:")
if duplicates_count > 0:
    print(f"   - {duplicates_count:,} SUPPLIER_IDs com duplicatas")
    print(f"   - {total_duplicate_records:,} registros duplicados no total")
    print(f"   - {initial_count - duplicates_count:,} registros únicos\n")
else:
    print(f"   - ✅ Nenhum duplicado encontrado!")
    print(f"   - {initial_count:,} registros únicos\n")

# 🔵 AUDITORIA: Análise de valores nulos por coluna
print(f"🟡 ANÁLISE DE VALORES NULOS:")
null_counts = bronze_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in bronze_df.columns
])

null_summary = null_counts.collect()[0].asDict()
for col_name, null_count in sorted(null_summary.items(), key=lambda x: x[1], reverse=True):
    null_pct = (null_count / initial_count) * 100
    if null_count > 0:
        print(f"   - {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")

# 🔵 AUDITORIA: Problemas específicos identificados
print(f"\n🔵 PROBLEMAS ESPECÍFICOS:")

# Postcode com decimais (postcode é DOUBLE)
postcode_with_decimal = bronze_df.filter(F.col("postcode").isNotNull()).filter(
    (F.col("postcode") % 1) != 0  # Se tem parte decimal
).count()
print(f"   - Postcode com decimais: {postcode_with_decimal:,} ({(postcode_with_decimal/initial_count)*100:.1f}%)")

# Number com letras
number_with_letters = bronze_df.filter(F.col("number").rlike("[A-Za-z]")).count()
print(f"   - Number alfanumérico: {number_with_letters:,} ({(number_with_letters/initial_count)*100:.1f}%)")

# District numérico (inválido)
district_numeric = bronze_df.filter(F.col("district").rlike("^[0-9.]+$")).count()
print(f"   - District numérico: {district_numeric:,} ({(district_numeric/initial_count)*100:.1f}%)")

# Coordenadas inválidas
invalid_coords = bronze_df.filter(
    (F.col("lon") < -180) | (F.col("lon") > 180) |
    (F.col("lat") < -90) | (F.col("lat") > 90)
).count()
print(f"   - Coordenadas inválidas: {invalid_coords:,}")

# Items provided vazios ou nulos
items_null = bronze_df.filter(F.col("items_provided").isNull() | (F.trim(F.col("items_provided")) == "")).count()
print(f"   - Items_provided vazios: {items_null:,} ({(items_null/initial_count)*100:.1f}%)")

print(f"\n📊 Exemplos de registros (primeiros 10):")
display(bronze_df.orderBy("SUPPLIER_ID").limit(10))

# 🧹 Etapa 3: Limpeza e Padronização de Tipos

Aplicar transformações para:
- Limpar e padronizar strings (TRIM, UPPER)
- Converter postcode de DOUBLE para STRING (removendo decimais)
- Ajustar tipos de dados
- Validar coordenadas geográficas
- Limpar items_provided

In [0]:
# Aplicar todas as transformações em uma única operação
transformed_df = bronze_df.select(
    # --- IDENTIFICAÇÃO ---
    # SUPPLIER_ID: Manter como INT (chave primária)
    F.col("SUPPLIER_ID"),
    
    # TAX_ID: Manter como INT
    F.col("TAX_ID"),
    
    # --- NOME ---
    # supplier_name: Limpar espaços e padronizar em maiúsculas
    F.trim(F.upper(F.col("supplier_name"))).alias("supplier_name"),
    
    # --- ENDEREÇO ---
    # state: Limpar espaços e padronizar
    F.trim(F.upper(F.col("state"))).alias("state"),
    
    # city: Limpar espaços e padronizar
    F.trim(F.upper(F.col("city"))).alias("city"),
    
    # postcode: Converter DOUBLE para STRING, remover decimais
    # Cast para BIGINT primeiro para remover decimais, depois para STRING
    F.when(
        F.col("postcode").isNotNull(),
        F.col("postcode").cast("bigint").cast("string")
    ).alias("postcode"),
    
    # street: Limpar espaços e padronizar
    F.trim(F.upper(F.col("street"))).alias("street"),
    
    # number: Manter como STRING (suporta alfanuméricos), apenas trim
    F.trim(F.col("number")).alias("number"),
    
    # unit: Limpar espaços
    F.trim(F.col("unit")).alias("unit"),
    
    # --- REGIÃO ---
    # region: Limpar e padronizar
    F.trim(F.upper(F.col("region"))).alias("region"),
    
    # district: Limpar valores numéricos puros (inválidos) e padronizar
    F.when(
        F.col("district").rlike("^[0-9.]+$"), 
        None
    ).otherwise(
        F.trim(F.upper(F.col("district")))
    ).alias("district"),
    
    # --- COORDENADAS ---
    # lon: Validar range (-180 a 180)
    F.when(
        (F.col("lon") >= -180) & (F.col("lon") <= 180),
        F.col("lon")
    ).otherwise(None).alias("lon"),
    
    # lat: Validar range (-90 a 90)
    F.when(
        (F.col("lat") >= -90) & (F.col("lat") <= 90),
        F.col("lat")
    ).otherwise(None).alias("lat"),
    
    # --- ITEMS PROVIDED ---
    # items_provided: Limpar espaços, padronizar
    F.trim(F.upper(F.col("items_provided"))).alias("items_provided")
)

print("✅ Transformações aplicadas com sucesso!")
print(f"\n📊 Novo schema após transformações:")
transformed_df.printSchema()

print(f"\n🔍 Amostra dos dados transformados (5 primeiras linhas):")
display(transformed_df.limit(5))

# 🗑️ Etapa 4: Remoção de Duplicados

Diferente de customers (que usa SCD Type 2), suppliers **não tem versionamento**.

Estratégia:
- Remover duplicados **completos** (todas as colunas iguais)
- Se houver duplicados parciais (mesmo SUPPLIER_ID, dados diferentes), manter o primeiro registro
- **Garantir** que cada SUPPLIER_ID apareça apenas uma vez na Silver

In [0]:
# Estratégia: Remover duplicados mantendo primeiro registro por SUPPLIER_ID
# Adicionar row_number para identificar duplicados
window_spec = Window.partitionBy("SUPPLIER_ID").orderBy(F.lit(1))

deduplicated_df = transformed_df.withColumn(
    "row_num",
    F.row_number().over(window_spec)
).filter(
    F.col("row_num") == 1
).drop("row_num")

# Contagem após remoção de duplicados
silver_count = deduplicated_df.count()
removed_duplicates = initial_count - silver_count

print("✅ Duplicados removidos com sucesso!")
print(f"\n🔵 AUDITORIA - REMOÇÃO DE DUPLICADOS:")
print(f"   - Registros na Bronze: {initial_count:,}")
print(f"   - Registros na Silver: {silver_count:,}")
print(f"   - Duplicados removidos: {removed_duplicates:,}")
print(f"   - Redução: {(removed_duplicates/initial_count)*100:.2f}%")

# Verificar se SUPPLIER_ID é único
unique_suppliers = deduplicated_df.select("SUPPLIER_ID").distinct().count()
print(f"\n✅ Verificação de unicidade:")
print(f"   - Total de registros: {silver_count:,}")
print(f"   - SUPPLIER_IDs únicos: {unique_suppliers:,}")
if silver_count == unique_suppliers:
    print(f"   - ✅ SUCESSO: Cada SUPPLIER_ID é único na Silver!")
else:
    print(f"   - ⚠️ ALERTA: Ainda existem duplicados!")

# ✅ Etapa 5: Validações Finais (PÓS-TRANSFORMAÇÃO)

Verificar a qualidade dos dados transformados:
- Confirmar unicidade de SUPPLIER_ID
- Analisar valores nulos após limpeza
- Validar ranges de coordenadas
- Comparar estatísticas antes/depois

In [0]:
print("🔵 AUDITORIA - VALIDAÇÕES PÓS-TRANSFORMAÇÃO")
print("="*60)

# 1. Confirmar unicidade de SUPPLIER_ID
print(f"\n1️⃣ UNICIDADE DE SUPPLIER_ID:")
duplicates_check = deduplicated_df.groupBy("SUPPLIER_ID").count().filter(F.col("count") > 1).count()
if duplicates_check == 0:
    print(f"   ✅ SUPPLIER_ID é único (0 duplicados)")
else:
    print(f"   ⚠️ {duplicates_check} SUPPLIER_IDs ainda estão duplicados!")

# 2. Análise de valores nulos após limpeza
print(f"\n2️⃣ VALORES NULOS APÓS LIMPEZA:")
null_counts_after = deduplicated_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c) 
    for c in deduplicated_df.columns
])

null_summary_after = null_counts_after.collect()[0].asDict()
for col_name, null_count in sorted(null_summary_after.items(), key=lambda x: x[1], reverse=True):
    null_pct = (null_count / silver_count) * 100
    if null_count > 0:
        print(f"   - {col_name}: {null_count:,} nulls ({null_pct:.2f}%)")

# 3. Validar coordenadas
print(f"\n3️⃣ VALIDAÇÃO DE COORDENADAS:")
valid_coords = deduplicated_df.filter(
    F.col("lon").isNotNull() & F.col("lat").isNotNull()
).count()
invalid_coords_after = deduplicated_df.filter(
    F.col("lon").isNull() | F.col("lat").isNull()
).count()
print(f"   - Coordenadas válidas: {valid_coords:,} ({(valid_coords/silver_count)*100:.1f}%)")
print(f"   - Coordenadas ausentes/inválidas: {invalid_coords_after:,} ({(invalid_coords_after/silver_count)*100:.1f}%)")

# 4. Análise de items_provided
print(f"\n4️⃣ ANÁLISE DE ITEMS_PROVIDED:")
items_filled = deduplicated_df.filter(F.col("items_provided").isNotNull()).count()
items_empty = deduplicated_df.filter(F.col("items_provided").isNull()).count()
print(f"   - Items preenchidos: {items_filled:,} ({(items_filled/silver_count)*100:.1f}%)")
print(f"   - Items vazios: {items_empty:,} ({(items_empty/silver_count)*100:.1f}%)")

print(f"\n" + "="*60)
print("✅ Validações concluídas!")

# 📝 Etapa 6: Adição de Colunas de Auditoria

Adicionar metadados para rastreabilidade:
- **processed_at**: Timestamp de quando foi processado
- **source_table**: Tabela de origem
- **pipeline_run_id**: Identificador único desta execução
- **data_quality_score**: Métrica de qualidade (% de campos não-nulos)

In [0]:
# Calcular data quality score para cada registro
# Score = (número de campos não-nulos / total de campos) * 100

data_columns = [
    "SUPPLIER_ID", "TAX_ID", "supplier_name", "state", "city",
    "postcode", "street", "number", "unit", "region", "district",
    "lon", "lat", "items_provided"
]

total_fields = len(data_columns)

# Contar campos não-nulos para cada registro
non_null_count = sum([
    F.when(F.col(c).isNotNull(), 1).otherwise(0) 
    for c in data_columns
])

# Adicionar colunas de auditoria
final_df = deduplicated_df.withColumn(
    "data_quality_score",
    F.round((non_null_count / total_fields) * 100, 2)
).withColumn(
    "processed_at",
    F.lit(processing_timestamp).cast("timestamp")
).withColumn(
    "source_table",
    F.lit(f"{CATALOG}.{BRONZE_SCHEMA}.{BRONZE_TABLE}")
).withColumn(
    "pipeline_run_id",
    F.lit(pipeline_run_id)
)

print("✅ Colunas de auditoria adicionadas!")
print(f"\n📊 Schema final com metadados:")
final_df.printSchema()

print(f"\n🔵 AUDITORIA - DATA QUALITY SCORE:")

# Distribuição de qualidade em buckets de 10%
quality_distribution = final_df.groupBy(
    (F.floor(F.col("data_quality_score") / 10) * 10).alias("quality_bucket")
).count().orderBy("quality_bucket")

print(f"\n📈 Distribuição de qualidade dos dados:")
display(quality_distribution)

avg_quality = final_df.agg(F.avg("data_quality_score")).collect()[0][0]
print(f"\n🎯 Score médio de qualidade: {avg_quality:.2f}%")

print(f"\n🔍 Amostra dos dados finais com auditoria:")
display(final_df.limit(5))

# 📦 Etapa 7: Escrita na Tabela Silver

Persistir os dados transformados na camada Silver:
- **Formato**: Delta Lake (suporte a ACID, Time Travel, Change Data Feed)
- **Modo**: Overwrite (primeira carga) / Merge (cargas incrementais)
- **Recursos Delta**: Habilitar Change Data Feed e otimizações

In [0]:
# Iniciar timer para medir tempo de escrita
import time
write_start = time.time()

# Escrever na tabela Silver com configurações Delta Lake
silver_table_name = f"{CATALOG}.{SILVER_SCHEMA}.{SILVER_TABLE}"

print(f"🔄 Iniciando escrita na tabela Silver...")
print(f"📍 Destino: {silver_table_name}")

# Escrever com Delta Lake
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(silver_table_name)

write_end = time.time()
write_duration = write_end - write_start

print(f"\n✅ Tabela Silver criada com sucesso!")
print(f"⏱️ Tempo de escrita: {write_duration:.2f} segundos")

# Otimizar a tabela (compactação de arquivos pequenos)
print(f"\n🛠️ Otimizando tabela...")
spark.sql(f"OPTIMIZE {silver_table_name}")
print("✅ Otimização concluída!")

# Coletar estatísticas da tabela
print(f"\n📊 Coletando estatísticas da tabela...")
spark.sql(f"ANALYZE TABLE {silver_table_name} COMPUTE STATISTICS")
print("✅ Estatísticas coletadas!")

# Verificar a tabela criada
print(f"\n🔍 Verificação da tabela Silver:")
silver_verification = spark.sql(f"SELECT COUNT(*) as count FROM {silver_table_name}").collect()[0][0]
print(f"   - Total de registros: {silver_verification:,}")

if silver_verification == silver_count:
    print(f"   ✅ Verificação bem-sucedida! Todos os {silver_count:,} registros foram gravados.")
else:
    print(f"   ⚠️ Divergência! Esperado: {silver_count:,}, Encontrado: {silver_verification:,}")

# 📊 Etapa 8: Relatório Final de Transformação

Resumo completo de todas as transformações aplicadas e métricas de auditoria.

In [0]:
from datetime import datetime

# Calcular tempo total de execução
processing_end = datetime.now()
total_duration = (processing_end - processing_timestamp).total_seconds()

print("\n" + "="*80)
print("🎆 RELATÓRIO FINAL DE TRANSFORMAÇÃO - BRONZE → SILVER 🎆")
print("="*80)

print(f"\n📅 INFORMAÇÕES DA EXECUÇÃO:")
print(f"   - Pipeline Run ID: {pipeline_run_id}")
print(f"   - Início: {processing_timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   - Término: {processing_end.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   - Duração total: {total_duration:.2f} segundos ({total_duration/60:.2f} minutos)")

print(f"\n📊 ESTATÍSTICAS DE REGISTROS:")
print(f"   - Registros na Bronze: {initial_count:,}")
print(f"   - Registros na Silver: {silver_count:,}")
print(f"   - Duplicados removidos: {removed_duplicates:,}")
print(f"   - Taxa de retenção: {(silver_count/initial_count)*100:.2f}%")

print(f"\n🎯 TRANSFORMAÇÕES APLICADAS:")
print(f"   ✅ Limpeza e padronização de strings (TRIM, UPPER)")
print(f"   ✅ Conversão de postcode (DOUBLE → STRING, sem decimais)")
print(f"   ✅ Validação de coordenadas geográficas")
print(f"   ✅ Limpeza de valores inválidos em district")
print(f"   ✅ Remoção de duplicados por SUPPLIER_ID")
print(f"   ✅ Adição de colunas de auditoria")
print(f"   ✅ Limpeza e padronização de items_provided")

print(f"\n📖 PROBLEMAS CORRIGIDOS:")
print(f"   - Postcode com decimais: {postcode_with_decimal:,} registros convertidos")
print(f"   - Number alfanumérico: {number_with_letters:,} mantidos como STRING")
print(f"   - District numérico: {district_numeric:,} valores limpos")
print(f"   - Coordenadas inválidas: {invalid_coords:,} valores anulados")

print(f"\n🔵 QUALIDADE DOS DADOS:")
avg_quality_final = final_df.agg(F.avg("data_quality_score")).collect()[0][0]
print(f"   - Score médio de qualidade: {avg_quality_final:.2f}%")
print(f"   - Suppliers únicos: {silver_count:,}")

print(f"\n💾 TABELA CRIADA:")
print(f"   - Nome: {silver_table_name}")
print(f"   - Formato: Delta Lake")
print(f"   - Change Data Feed: Habilitado")
print(f"   - Registros: {silver_verification:,}")

print(f"\n📄 COLUNAS ADICIONADAS:")
print(f"   - data_quality_score: Métrica de qualidade (% não-nulos)")
print(f"   - processed_at: Timestamp de processamento")
print(f"   - source_table: Tabela de origem")
print(f"   - pipeline_run_id: ID único da execução")

print(f"\n📝 DIFERENÇAS EM RELAÇÃO A CUSTOMERS:")
print(f"   - ❌ SEM SCD Type 2 (sem valid_from/valid_to, is_current)")
print(f"   - ✅ Lógica mais simples: duplicados removidos diretamente")
print(f"   - ✅ Coluna específica: items_provided")

print(f"\n" + "="*80)
print("✨ TRANSFORMAÇÃO CONCLUÍDA COM SUCESSO! ✨")
print("="*80)

# Criar DataFrame de sumário
summary_data = [
    ["Registros Bronze", str(initial_count)],
    ["Registros Silver", str(silver_count)],
    ["Duplicados Removidos", str(removed_duplicates)],
    ["Taxa de Retenção (%)", f"{round((silver_count/initial_count)*100, 2)}%"],
    ["Score Médio de Qualidade (%)", f"{round(avg_quality_final, 2)}%"],
    ["Tempo de Execução (seg)", str(round(total_duration, 2))]
]

summary_df = spark.createDataFrame(summary_data, ["Métrica", "Valor"])

print(f"\n📊 Dashboard de Auditoria:")
display(summary_df)